In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import count
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("AirbnbBigData") \
    .config("spark.driver.memory", "4g") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

listings_schema = StructType([
    StructField("listing_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("room_type", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("minimum_nights", IntegerType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("review_scores_rating", DoubleType(), True)
])

reviews_schema = StructType([
    StructField("listing_id", IntegerType(), True),
    StructField("review_id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("reviewer_id", IntegerType(), True)
])

listings_df = spark.read.csv("../data/cleaned/listings_clean.csv", header=True, schema=listings_schema)
reviews_df = spark.read.csv("../data/cleaned/reviews_clean.csv", header=True, schema=reviews_schema)

review_metrics = reviews_df.groupBy("listing_id").agg(
    count("review_id").alias("total_reviews")
)

enriched_listings = listings_df.join(review_metrics, on="listing_id", how="left")
enriched_listings = enriched_listings.na.fill(0, ["total_reviews"])

print("Aggregated Big Data sample:")
enriched_listings.select("listing_id", "city", "price", "total_reviews").show(5)

final_df = enriched_listings.toPandas()
os.makedirs("../data/cleaned/listings_with_metrics", exist_ok=True)
final_df.to_csv("../data/cleaned/listings_with_metrics/data.csv", index=False)
print("Pipeline finished successfully!")

spark.stop()

Aggregated Big Data sample:
+----------+-----+-----+-------------+
|listing_id| city|price|total_reviews|
+----------+-----+-----+-------------+
|   4082273|Paris| 89.0|            1|
|   4823489|Paris| 60.0|            1|
|   4898654|Paris| 95.0|            2|
|    281420|Paris| 53.0|            2|
|   4797344|Paris| 58.0|            1|
+----------+-----+-----+-------------+
only showing top 5 rows
Pipeline finished successfully!
